In [ ]:
from mofdb_client import fetch

In [ ]:
import re
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from mofdb_client import fetch

In [ ]:
# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------

# Pressure points present in hMOF GCMC data (bar)
CO2_PRESSURES = [0.01, 0.05, 0.1, 0.5, 2.5]
PRESSURE_TOL  = 1e-3   # absolute tolerance for pressure matching (bar)

def _col(p: float) -> str:
    """Column name for a CO2 pressure point, e.g. 'co2_mol_kg_0.05bar'."""
    return f"co2_mol_kg_{p}bar"


def _parse_topology(mofid: str | None) -> str | None:
    """
    Extract net topology from a MOF-ID string.
    MOF-ID format: '<smiles_fragment(s)> MOFid-v1.<topology>.<cat>'
    """
    if not mofid:
        return None
    m = re.search(r"MOFid-v1\.([^.\s]+)", mofid)
    return m.group(1) if m else None


def _parse_metal_node(mofkey: str | None) -> str | None:
    """
    Extract metal-node label from a MOF-key string.
    MOF-key format: '<Metal>.<InChIKey1>.<InChIKey2>...MOFkey-v1.<topology>'
    The first token before the first '.' is the metal symbol(s).
    """
    if not mofkey:
        return None
    return mofkey.split(".")[0]


def _extract_co2_uptake(mof) -> dict:
    """
    Return a dict {col_name: uptake_mol_kg} for all CO2 pressure points.
    Picks the 298 K pure-CO2 simulation isotherm; falls back to any CO2 isotherm.
    If multiple isotherms exist, prefers simulated (simin='True') ones.
    """
    uptake = {_col(p): np.nan for p in CO2_PRESSURES}

    # Filter to CO2-only isotherms at 298 K
    co2_isos = [
        iso for iso in mof.isotherms
        if len(iso.adsorbates) == 1
        and iso.adsorbates[0].name == "CarbonDioxide"
        and abs(iso.temperature - 298.0) < 1.0
        and iso.pressureUnits.lower() in ("bar",)
        and "mol/kg" in iso.adsorptionUnits.lower()
    ]

    # Prefer simulated isotherms
    sim_isos = [iso for iso in co2_isos if str(iso.simin).lower() == "true"]
    chosen = sim_isos[0] if sim_isos else (co2_isos[0] if co2_isos else None)

    if chosen is None:
        return uptake

    for pt in chosen.isotherm_data:
        for target_p in CO2_PRESSURES:
            if abs(pt.pressure - target_p) < PRESSURE_TOL:
                # total_adsorption is the sum; for pure-CO2 it equals species_data[0].adsorption
                uptake[_col(target_p)] = pt.total_adsorption
                break

    return uptake


def _mof_to_row(mof) -> dict:
    """Convert a Mof object into a flat dict suitable for a DataFrame row."""
    sa_m2g   = mof.surface_area_m2g
    sa_m2cm3 = mof.surface_area_m2cm3

    # Density: rho = SA[m²/cm³] / SA[m²/g]  (algebraically: g/cm³)
    if sa_m2g and sa_m2cm3 and sa_m2g > 0:
        density = sa_m2cm3 / sa_m2g
    else:
        density = np.nan

    row = {
        "name"               : mof.name,
        "mofdb_id"           : mof.id,
        "database"           : mof.database,
        "pld"                : mof.pld,
        "lcd"                : mof.lcd,
        "surface_area_m2g"   : sa_m2g,
        "surface_area_m2cm3" : sa_m2cm3,
        "void_fraction"      : mof.void_fraction,
        "density_g_cm3"      : density,
        "topology"           : _parse_topology(mof.mofid),
        "metal_node"         : _parse_metal_node(mof.mofkey),
        "elements"           : ",".join(str(e) for e in mof.elements),
        "mofid"              : mof.mofid,
        "mofkey"             : mof.mofkey,
    }
    row.update(_extract_co2_uptake(mof))
    return row

# hMOF Dataset Download & Assembly

Download the full hMOF dataset (~130,000 structures) from [MOFDB](https://mof.tech.northwestern.edu) via `mofdb_client`.

**Descriptors extracted per structure:**
| Field | Description |
|---|---|
| `pld` | Pore-limiting diameter (Å) |
| `lcd` | Largest-cavity diameter (Å) |
| `surface_area_m2g` | Accessible surface area (m²/g) |
| `surface_area_m2cm3` | Accessible surface area (m²/cm³) |
| `void_fraction` | Helium void fraction / porosity |
| `density_g_cm3` | Crystal density (g/cm³), derived as SA[m²/cm³] / SA[m²/g] |
| `topology` | Net topology parsed from MOF-ID (e.g. `pcu`, `dia`) |
| `metal_node` | Metal symbol(s) parsed from MOF-key |
| `elements` | Full element list |

**CO2 uptake columns** (298 K, GCMC, mol/kg):  
The hMOF MOFDB entries contain 5 pressure points: **0.01, 0.05, 0.1, 0.5, 2.5 bar**.  
> Note: the originally requested pressures were 0.05, 0.15, 0.5, 1.0, 2.5 bar. The 0.15 bar and 1.0 bar points are not available in the MOFDB hMOF dataset; the dataset contains 0.01 bar and 0.1 bar instead. All five available pressure points are stored as `co2_mol_kg_<P>bar` columns.